# Space-time accessibility results

In [2]:
%load_ext autoreload
%autoreload 2
%cd D:\netmob25

D:\netmob25


In [3]:
import os
import pandas as pd
import glob
from tqdm import tqdm

## 1. Load commuters' activity data

In [4]:
df = pd.read_csv('dbs/data_p/commuter_trips.csv')
print(df['dow'].unique())
# Update to consider only weekdays for now (SPA alignment consideration)
# df = df[~df['dow'].isin(['saturday', 'sunday'])]
# df.head()

['wednesday' 'thursday' 'friday' 'monday' 'tuesday' 'saturday' 'sunday']


In [5]:
df_tt = pd.read_csv("dbs/data_p/commuter_time_budget.csv")
# df_tt.head()

In [6]:
# Individual - day results
df_r = pd.read_csv("dbs/data_p/trip_chaining_results.csv")
# df_r.head()

## 2. Process accessibile locations' time requirement

In [21]:
mode_list = ['pt', 'car']
situation_list = ['01', '02', '03']
departure_time = '17'
# Folder
folder = "dbs/sp_accessibility/"
df_sp_ind_list = []
for mode in mode_list:
    for situation in situation_list:
        # time wk
        all_files = []
        file_name_head = f"tt_wk_{mode}_{departure_time}_{situation}_"
        pattern = os.path.join(folder, file_name_head + "*.csv")
        all_files.extend(glob.glob(pattern))
        # Load into one DataFrame
        df_sp = pd.concat((pd.read_csv(f) for f in all_files),
                    ignore_index=True)
        df_sp.rename(columns={'travel_time_p50': 'time_wk', 'from_id': 'ID', 'to_id': 'poi_id'}, inplace=True)
        #print(df_sp.shape)
        #print(df_sp.head())
        
        # time kh
        all_files = []
        file_name_head = f"tt_kh_{mode}_{departure_time}_{situation}_"
        pattern = os.path.join(folder, file_name_head + "*.csv")
        all_files.extend(glob.glob(pattern))
        df_kh = pd.concat((pd.read_csv(f) for f in all_files),
                    ignore_index=True)
        df_kh.rename(columns={'travel_time_p50': 'time_kh', 'from_id': 'poi_id', 'to_id': 'ID'}, inplace=True)
        # print(df_kh.shape)
        #print(df_kh.head())
        
        # Merge
        df_sp = df_sp.merge(df_kh, on=['ID', 'poi_id'], how='inner')
        del df_kh
        #print(df_sp.shape)
        #print(df_sp.head())
        
        # Time budget
        df_sp = df_sp.merge(df_tt[['ID', f'tt_wkh_{int(situation)}']].rename(columns={f'tt_wkh_{int(situation)}': 'tt_wkh'}), on='ID', how='left')
        df_sp['time_left'] = df_sp['tt_wkh'] - df_sp['time_wk'] - df_sp['time_kh']
        tqdm.pandas(desc="Processing accessibility data")
        df_sp_ind = df_sp.groupby('ID').progress_apply(lambda x: pd.Series(dict(ak = x.loc[x['time_left'] >= 0, 'poi_id'].nunique()))).reset_index()
        df_sp_ind['mode'] = mode
        df_sp_ind['situation'] = situation
        df_sp_ind_list.append(df_sp_ind)

Processing accessibility data: 100%|██████████| 2047/2047 [00:06<00:00, 330.59it/s]


In [22]:
df_sp_ind = pd.concat(df_sp_ind_list, ignore_index=True)
df_sp_ind.head()

,ID,ak,mode,situation
0,10_2978,8385,pt,01
1,10_2995,0,pt,01
2,10_2998,0,pt,01
3,10_3000,0,pt,01
4,10_3003,54,pt,01


In [27]:
# Step 1: get the complete set of IDs, modes, and situations
all_ids = df_tt["ID"].unique()
all_modes = df_sp_ind["mode"].unique()
all_situations = df_sp_ind["situation"].unique()

# Step 2: create full index combinations
full_index = pd.MultiIndex.from_product(
    [all_ids, all_modes, all_situations],
    names=["ID", "mode", "situation"]
)

# Step 3: set index and reindex
df_complete = (
    df_sp_ind
    .set_index(["ID", "mode", "situation"])
    .reindex(full_index, fill_value=0)   # missing ak → 0
    .reset_index()
)
df_complete = df_complete[df_complete['situation'] != '03']  # remove situation 03
df_complete.head()

,ID,mode,situation,ak
0,10_2978,pt,01,8385
1,10_2978,pt,02,0
3,10_2978,car,01,19245
4,10_2978,car,02,401
6,10_2980,pt,01,0


In [28]:
print(df_complete.ID.nunique())

2456


In [29]:
df_complete.to_csv("dbs/data_p/commuter_sp_accessibility.csv", index=False)

## 2r. Process accessible locations

In [ ]:
mode_list = ['pt', 'car']
departure_time = '17'
# Folder
folder = "dbs/sp_accessibility_r/"
df_sp_ind_list = []
df_sp_set_list = []
for mode in mode_list:
    # time wk
    all_files = []
    file_name_head = f"tt_wk_{mode}_{departure_time}_"
    pattern = os.path.join(folder, file_name_head + "*.csv")
    all_files.extend(glob.glob(pattern))
    # Load into one DataFrame
    df_sp = pd.concat((pd.read_csv(f) for f in all_files),
                ignore_index=True)
    df_sp.rename(columns={'travel_time_p50': 'time_wk', 'from_id': 'ID', 'to_id': 'poi_id'}, inplace=True)
    #print(df_sp.shape)
    #print(df_sp.head())
    
    # time kh
    all_files = []
    file_name_head = f"tt_kh_{mode}_{departure_time}_"
    pattern = os.path.join(folder, file_name_head + "*.csv")
    all_files.extend(glob.glob(pattern))
    df_kh = pd.concat((pd.read_csv(f) for f in all_files),
                ignore_index=True)
    df_kh.rename(columns={'travel_time_p50': 'time_kh', 'from_id': 'poi_id', 'to_id': 'ID'}, inplace=True)
    # print(df_kh.shape)
    #print(df_kh.head())
    
    # Merge
    df_sp = df_sp.merge(df_kh, on=['ID', 'poi_id'], how='inner')
    del df_kh
    #print(df_sp.shape)
    #print(df_sp.head())
    
    # Time budget
    situation = '02'
    df_sp = df_sp.merge(df_tt[['ID', f'tt_wkh_{int(situation)}']].rename(columns={f'tt_wkh_{int(situation)}': 'tt_wkh'}), on='ID', how='left')
    df_sp['time_left'] = df_sp['tt_wkh'] - df_sp['time_wk'] - df_sp['time_kh']
    tqdm.pandas(desc="Processing accessibility data")
    df_sp_ind = df_sp.groupby('ID').progress_apply(lambda x: pd.Series(dict(ak = x.loc[x['time_left'] >= 0, 'poi_id'].nunique()))).reset_index()
    df_sp_set = df_sp.loc[df_sp['time_left']>0, ['ID', 'poi_id', 'time_left']].copy()

    df_sp_ind['mode'] = mode
    df_sp_set['mode'] = mode
    df_sp_ind_list.append(df_sp_ind)
    df_sp_set_list.append(df_sp_set)

Processing accessibility data: 100%|██████████| 1416/1416 [00:28<00:00, 49.77it/s]


In [9]:
df_sp_ind = pd.concat(df_sp_ind_list, ignore_index=True)
df_sp_ind.head()

,ID,ak,mode
0,10_2978,0,pt
1,10_2980,10934,pt
2,10_2984,87,pt
3,10_2993,7,pt
4,10_2994,0,pt


In [10]:
df_sp_set = pd.concat(df_sp_set_list, ignore_index=True)
df_sp_set.head()

,ID,poi_id,time_left,mode
0,16_4000,08f1fb4670bb338a03a0f7c1eb8d83f2,2.0,pt
1,16_4000,08f1fb4670bb338a03a0f7c1eb8d83f2,2.0,pt
2,16_4000,08f1fb4670bb338a03a0f7c1eb8d83f2,2.0,pt
3,16_4000,08f1fb4670bb338a03a0f7c1eb8d83f2,2.0,pt
4,16_4000,08f1fb4670bb338a03a0f7c1eb8d83f2,2.0,pt


In [10]:
# Step 1: get the complete set of IDs, modes, and situations
all_ids = df_tt["ID"].unique()
all_modes = df_sp_ind["mode"].unique()

# Step 2: create full index combinations
full_index = pd.MultiIndex.from_product(
    [all_ids, all_modes],
    names=["ID", "mode"]
)

# Step 3: set index and reindex
df_complete = (
    df_sp_ind
    .set_index(["ID", "mode"])
    .reindex(full_index, fill_value=0)   # missing ak → 0
    .reset_index()
)
df_complete.head()

,ID,mode,ak
0,10_2978,pt,0
1,10_2978,car,6528
2,10_2980,pt,10934
3,10_2980,car,21405
4,10_2981,pt,0


In [11]:
print(df_complete.ID.nunique())
df_complete.to_csv("dbs/data_p/commuter_sp_accessibility_r.csv", index=False)

2456


In [11]:
print(df_sp_set.ID.nunique())
df_sp_set.to_parquet("dbs/data_p/commuter_sp_set_r.parquet", index=False)

1144


In [13]:
df_sp_set['time_left'].describe()

count    6.454220e+07
mean     1.772493e+01
std      1.312128e+01
min      1.000000e+00
25%      7.000000e+00
50%      1.500000e+01
75%      2.600000e+01
max      8.500000e+01
Name: time_left, dtype: float64

## 3. Individual attributes

In [ ]:
df_ind = pd.read_parquet("results/activity_access_ind_model.parquet")
df_ind.columns

In [32]:
df_ind = df_ind[['ID', 'time_threshold', 'amenity', 'mode', 'access_h', 'codgeo', 'weight_ind', 
                 'Gender', 'Age', 'Education', 'Household_type', 
                 'Car_no', 'Bike_no', 'Two_wheeler_no', 'Escooter_no', 'pt_sub', 'main_mode']]

In [56]:
df_ind[df_ind.ID.isin(df_complete.ID.unique())].to_csv("dbs/data_p/commuter_attributes.csv", index=False)

## 4. Mobility efficiency results join (individual-day)

In [8]:
# Add main mode per trip/day
df_raw = pd.read_csv('dbs/data/trips_dataset.csv')
df_raw = df_raw[['ID', 'Main_Mode', 'Day_EMG', 'Date_EMG', 'Day_Type', 'Weight_Day']]
df_raw.rename(columns={'Main_Mode': 'main_mode_day', 'Day_EMG': 'dow', 'Date_EMG': 'date', 'Day_Type': 'day_type', 'Weight_Day': 'weight_day'}, inplace=True)
df_raw.head()

,ID,main_mode_day,dow,date,day_type,weight_day
0,42_0001,NaN,monday,2022-10-17,Normal,97.171946
1,42_0001,PRIV_CAR_PASSENGER,tuesday,2022-10-18,Strike,92.090337
2,42_0001,PRIV_CAR_PASSENGER,tuesday,2022-10-18,Strike,92.090337
3,42_0001,PRIV_CAR_PASSENGER,tuesday,2022-10-18,Strike,92.090337
4,42_0001,PRIV_CAR_PASSENGER,tuesday,2022-10-18,Strike,92.090337


In [14]:
def weighted_mode(values, weights):
    w = weights.groupby(values).sum()
    return w.idxmax()

df_day = (
    df_raw.dropna(subset=['main_mode_day']).sort_values(['ID', 'date'])
      .groupby(['ID', 'date'], as_index=False)
      .apply(lambda g: pd.Series({
          'dow': g['dow'].iloc[0],
          'day_type': g['day_type'].iloc[0],
          'weight_day': g['weight_day'].iloc[0],
          'main_mode_day': weighted_mode(g['main_mode_day'], g['weight_day'])
      }), include_groups=False)
      .reset_index(drop=True)
)

In [17]:
df_rind = pd.merge(df_r, df_day, on=['ID', 'date'], how='left')
df_rind.drop(columns=['freq_hwsx', 'xs_total_hwsx'], inplace=True)
print(df_rind.columns)

Index(['ID', 'date', 'trip_chaining_presence', 'freq_hws', 'xs_total_hws',
       'total_travel_time', 'activity_time_third', 'activity_time_work_study',
       'activity_time_home', 'entropy_naive', 'entropy_mm', 'gini_simpson',
       'hill_q1', 'n_visits', 'activity_nh_ratio', 'dow', 'day_type',
       'weight_day', 'main_mode_day'],
      dtype='object')


In [19]:
df_complete = pd.read_csv("dbs/data_p/commuter_sp_accessibility_r.csv")

In [20]:
df_rind[df_rind['ID'].isin(df_complete['ID'].unique())].to_csv("dbs/data_p/commuter_mobi_efficiency_daily.csv", index=False)

In [21]:
print(df_rind.iloc[0])

ID                                  10_2978
date                             2023-03-15
trip_chaining_presence                  0.0
freq_hws                                0.0
xs_total_hws                            0.0
total_travel_time                      55.0
activity_time_third                     0.0
activity_time_work_study                0.0
activity_time_home                   1385.0
entropy_naive                           NaN
entropy_mm                              NaN
gini_simpson                            NaN
hill_q1                                 NaN
n_visits                                0.0
activity_nh_ratio                       0.0
dow                               wednesday
day_type                             Normal
weight_day                       235.905683
main_mode_day               PRIV_CAR_DRIVER
Name: 0, dtype: object
